In [ ]:
import numpy as np
import sys
import os

# Add the project root to the path
sys.path.append('/home/justin/code/point-to-pose')

# from point2pose.modules.register.svd_register import SVDRegister
# from point2pose.modules.register.svd_residual_outlier import SVDResidualOutlierRegister
from point2pose.utils.transform import transform_pts, inverse_SE3

# ---- 1) Load the whole npz ----
D = np.load('/home/justin/code/point-to-pose/debug/pipeline/meta_data/meata_data.npz', allow_pickle=True)  # dict-like

print("Keys:", list(D.files))  # discover what's inside
N = len(D["frame_id"])        # number of rows/frames
print("Num rows:", N)

# ---- 2) Helper to unpack ragged fields ----
def unpack_ragged(name: str, store: dict, dim=-1):
    data    = store[f"{name}_data"]
    offsets = store[f"{name}_offsets"]
    lengths = store[f"{name}_lengths"]
    out = []
    for off, L in zip(offsets, lengths):
        flat_data = data[off:off+L]
        # Reshape to (N, 3) assuming 3D points
        if dim == 3:
            reshaped_data = flat_data.reshape(-1, 3)
        elif dim == 2:
            reshaped_data = flat_data.reshape(-1, 2)
        elif dim == -1:
            reshaped_data = flat_data
        else:
            print(f"Warning: {name} data length {len(flat_data)} not divisible by 3")
            reshaped_data = flat_data  # Keep as 1D if can't reshape
        out.append(reshaped_data)
    return out  # -> list of (N, 3) ndarrays (one per row)

# ---- 3) Access fixed-shape fields (already stacked) ----
timestamp  = D["timestamp"]          # shape (N,)
frame_id   = D["frame_id"]           # shape (N,)

print(D["reg_key_points_data"].shape)

# ---- 4) Access ragged fields ----
reg_key_points_idx_list = unpack_ragged("reg_key_points_idx", D)  # list of (Mi,) int arrays
reg_key_points_list = unpack_ragged("reg_key_points", D,dim=3)  # list of (Mi,3) float arrays
reg_cur3d_list = unpack_ragged("reg_curr3d", D,dim=3)            # list of (Mi,3) float arrays
reg_inlier_list = unpack_ragged("reg_inliers", D)          # list of (Mi,) bool arrays
reg_residual_list = unpack_ragged("reg_residuals", D)      # list of (Mi,) float arrays
track3d = unpack_ragged("track3d", D,dim=3)
visibles = unpack_ragged("visibles", D)
uncertainties = unpack_ragged("uncertainties", D)


# obj pose
obj_init_pose = D["obj_init_pose"][0]
obj_pose_list = D["obj_pose"]
obj_key_points = unpack_ragged("obj_key_points", D,dim=3)
obj_uncertainties = unpack_ragged("obj_uncertainties", D)


print(len(obj_key_points))



print(f"\nExtracted registration data:")
print(f"  reg_key_points_list: {len(reg_key_points_list)} frames")
print(f"  reg_cur3d_list: {len(reg_cur3d_list)} frames")

# Show shapes for first few frames
# for i in range(min(3, len(reg_key_points_list))):
#     print(f"  Frame {i}: reg_key_points {reg_key_points_list[i].shape}, reg_cur3d {reg_cur3d_list[i].shape}")
#     print(reg_key_points_list[i])


# Create a register instance for debugging
config = {
    'debug_level': 1,
    'debug_dir': '/home/justin/code/point-to-pose/debug/register_test'
}

print(f"\nCreated SVDRegister instance with debug level: {config['debug_level']}")

# Store the data for use in other cells
print(f"\nData loaded successfully! Available variables:")
print(f"  - D: Full data dictionary")
print(f"  - N: Number of frames ({N})")
print(f"  - frame_id, obj_id, res_mean, num_points: Fixed-shape arrays")
print(f"  - reg_key_points_list, reg_cur3d_list: Lists of point clouds")


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib
# Enable interactive mode for 3D plots
matplotlib.use('TkAgg')  # or 'Qt5Agg' depending on your system
plt.ion()  # Turn on interactive mode

def visualize_pcd(pcd):
    # Get points from your point cloud
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None

    # Create 3D plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot points
    if colors is not None:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                c=colors, s=1, alpha=0.8)
    else:
        ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                c='blue', s=1, alpha=0.8)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('Point Cloud Visualization')
    plt.show()

In [ ]:
import gtsam
import time

prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1]))
between_noise = gtsam.noiseModel.Diagonal.Sigmas(
    np.array([0.01, 0.01, 0.01, 0.01, 0.01, 0.01])
)

graph = gtsam.NonlinearFactorGraph()
initial_estimate = gtsam.Values()

inserted_landmarks = set()
prev_num_kp = 0
t = time.time()
for i in range(len(obj_pose_list)):
# t = time.time()
# for i in range(50):

    Xi = gtsam.symbol('x',i)

    residuals = reg_residual_list[i]

    if i == 0:
        X0 = gtsam.Pose3(inverse_SE3(obj_init_pose))
        initial_estimate.insert(Xi, X0)

        graph.push_back(gtsam.PriorFactorPose3(Xi, X0, prior_noise))

        current_estimate = initial_estimate
    else:

        Xim1 = gtsam.symbol('x',i-1)
        
        pose_i = inverse_SE3(obj_pose_list[i])
        pose_im1_inv = obj_pose_list[i - 1]
        # add initial guess for the pose
        initial_estimate.insert(Xi, gtsam.Pose3(pose_i))

        # add between factor
        between_noise = gtsam.noiseModel.Isotropic.Sigma(6, np.mean(residuals))
        between_pose = gtsam.Pose3(pose_im1_inv @ pose_i)
        graph.push_back(
            gtsam.BetweenFactorPose3(Xim1, Xi, between_pose, between_noise)
        )

    # add landmark
    cur_kp = obj_key_points[i]
    cur_kp_idx = reg_key_points_idx_list[i]
    cur_3d = reg_cur3d_list[i]
    inliers = reg_inlier_list[i]
    # print(inliers)
    # if len(cur_kp) > prev_num_kp:
    #     for kp in cur_kp:
    #         if kp not in inserted_landmarks:
    #             inserted_landmarks.add(kp)
    #             graph.push_back(gtsam.PriorFactorPoint3(gtsam.symbol('l', len(inserted_landmarks) - 1), kp, prior_noise))
    #     prev_num_kp = len(cur_kp)

    # # add between factor
    

    # for kp in cur_3d:
    point_noise       = gtsam.noiseModel.Isotropic.Sigma(3, 0.01)
    # Loop over measurements
    for m, lid in enumerate(cur_kp_idx):
        if inliers.size == 0:
            continue
        if not inliers[m]:
            continue
        cur_kp_i = cur_kp[m]
        z_cam = cur_3d[m]                          # (3,), camera_i frame
        Lj = gtsam.symbol('l', int(lid))                 # stable landmark key by your ID
        
        Xi_ = obj_pose_list[i]

        Xi_init = initial_estimate.atPose3(Xi)
        z_cam_world = Xi_init.transformFrom(gtsam.Point3(*z_cam))
        # print("Xi_init", Xi_init)
        # print("cur_kp_i", cur_kp_i)
        # print("z_cam_world", z_cam_world)
        # print("z_cam", z_cam)
        # Seed landmark the first time we see it
        if not initial_estimate.exists(Lj):
            # world guess from current pose + camera measurement: p_w ≈ X_i * z_cam
            Xi_init = initial_estimate.atPose3(Xi)
            p_w = Xi_init.transformFrom(gtsam.Point3(*z_cam))
            initial_estimate.insert(Lj, p_w)

        
        # compute range and bearing
        z_range = np.linalg.norm(z_cam)
        z_bearing = gtsam.Unit3(z_cam / z_range)

        # add factor
        graph.push_back(gtsam.BearingRangeFactor3D(Xi, Lj, z_bearing, z_range, point_noise))
print("adding time ", time.time() - t)
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimate, params)

t_before_optimizing = time.time()
# Perform the optimization
result = optimizer.optimize()
print("full optimizing time ", time.time() - t)
print("optimizing time ", time.time() - t_before_optimizing)
# print(result)

# print("\nFactor Graph:\n{}".format(graph))


# marginals = gtsam.Marginals(graph, current_estimate)
# i = 0
# while current_estimate.exists(gtsam.symbol('x',i)):
#     print(f"X{i} covariance:\n{marginals.marginalCovariance(gtsam.symbol('x',i))}\n")
#     i += 1



In [ ]:
## ISAM2 version 

# --- noise models you already had ---
prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1]))

# iSAM2 setup
isam_params = gtsam.ISAM2Params()
# (optional but common tunings)
isam_params.setRelinearizeThreshold(0.1)  # per-variable threshold
isam_params.relinearizeSkip = 1         # check every update
isam = gtsam.ISAM2(isam_params)

# Keep track of what we've already initialized
inserted_poses = set()
inserted_landmarks = set()


new_graph = gtsam.NonlinearFactorGraph()    
new_values = gtsam.Values()

t_all = time.time()
for i in range(len(obj_pose_list)):
    Xi = gtsam.symbol('x', i)
    
    
    
    residuals = reg_residual_list[i]
    # A safe scalar for between noise (avoid 0)
    sigma_between = float(max(1e-4, np.mean(residuals))) if residuals.size else 0.01
    between_noise = gtsam.noiseModel.Isotropic.Sigma(6, sigma_between)
    # between_noise = gtsam.noiseModel.Isotropic.Sigma(6, 0.01)


    # Pose initialization from your measurements (you already do inverse_SE3)
    pose_i_SE3 = gtsam.Pose3(inverse_SE3(obj_pose_list[i]))

    # Insert Xi only once (iSAM2 keeps linearization point internally)
    if Xi not in inserted_poses:
        new_values.insert(Xi, pose_i_SE3)
        inserted_poses.add(Xi)

    # Add prior once, at i = 0
    if i == 0:
        print(pose_i_SE3)
        new_graph.push_back(gtsam.PriorFactorPose3(Xi, pose_i_SE3, prior_noise))
    else:
        Xim1 = gtsam.symbol('x', i - 1)

        # Relative pose (you were using pose_im1_inv @ pose_i)
        pose_im1_inv = obj_pose_list[i - 1]     # "inverse" in your naming; keep as-is
        rel_T = gtsam.Pose3(pose_im1_inv @ inverse_SE3(obj_pose_list[i]))
        new_graph.push_back(gtsam.BetweenFactorPose3(Xim1, Xi, rel_T, between_noise))

    # Landmark/bearing-range factors (only inliers)
    cur_kp      = obj_key_points[i]
    cur_kp_idx  = reg_key_points_idx_list[i]
    cur_3d      = reg_cur3d_list[i]
    inliers     = reg_inlier_list[i]
    uncertainties = obj_uncertainties[i]

    print("cur_kp shape: ", cur_kp.shape)
    print("uncertainties shape: ", uncertainties.shape)

    # point_noise = gtsam.noiseModel.Isotropic.Sigma(3, np.mean(residuals))

    # We'll use the *current* initial/estimated pose to seed new landmarks
    # If we haven't run iSAM yet, fall back to our inserted guess
    if i == 0:
        Xi_seed = pose_i_SE3
    else:
        # current estimate if available; otherwise the inserted guess
        try:
            Xi_seed = isam.calculateEstimate().atPose3(Xi)
        except RuntimeError:
            Xi_seed = pose_i_SE3

    # Loop over measurements
    if inliers.size > 0:
        for m, lid in enumerate(cur_kp_idx):
            if not inliers[m]:
                continue
            z_cam = cur_3d[m]                    # 3D point in camera_i frame (your convention)
            Lj = gtsam.symbol('l', int(lid))
            point_noise = gtsam.noiseModel.Isotropic.Sigma(3, residuals[m])
            # Seed landmark once (in world), using Xi_seed * z_cam
            if Lj not in inserted_landmarks:
                p_w = Xi_seed.transformFrom(gtsam.Point3(*z_cam))
                new_values.insert(Lj, p_w)
                inserted_landmarks.add(Lj)

            # form bearing/range in the camera/body frame (your code does this)
            z_range = float(np.linalg.norm(z_cam))
            if z_range <= 1e-9:
                continue
            z_bearing = gtsam.Unit3(z_cam / z_range)

            new_graph.push_back(
                gtsam.BearingRangeFactor3D(Xi, Lj, z_bearing, z_range, point_noise)
            )

    # --- Incremental update ---
    isam.update(new_graph, new_values)

    # (Optional) occasionally force an extra relinearization sweep
    # if i % 10 == 0:
    #     isam.update()
    if i == 0 or i == 1:
        print(f"[ISAM2Optimizer] frame id: {i}")
        print(f"[ISAM2Optimizer] Graph: {new_graph}")
        print(f"[ISAM2Optimizer] Values: {new_values}")

    points = gtsam.utilities.extractPoint3(result)
    print(type(points))
    print(points.shape)

    # Get rolling estimate if you want to use it online
    current_estimate = isam.calculateEstimate()
    # Example: read back pose_i now
    # Xi_hat = current_estimate.atPose3(Xi)
    # print(f"Step {i} pose: \n{Xi_hat}")

    new_graph.resize(0)
    new_values.clear()

print("iSAM2 total wall time:", time.time() - t_all)

t_before_final_result = time.time()
# Final result
result = isam.calculateEstimate()
print("final result time ", time.time() - t_before_final_result)

poses = gtsam.utilities.extractPose3(result)
print(type(poses))
print(poses.shape)
points = gtsam.utilities.extractPoint3(result)
print(type(points))
print(points.shape)
# Example: access poses/landmarks
# for i in range(len(obj_pose_list)):
#     Xi = gtsam.symbol('x', i)
#     if result.exists(Xi):
#         print("Pose", i, "=\n", result.atPose3(Xi))

# for lid in inserted_landmarks:
#     if result.exists(lid):
#         print("Landmark", gtsam.Symbol(lid).index(), "=", result.atPoint3(lid))

In [ ]:
# import graphviz

# display(graphviz.Source(graph.dot(initial_estimate)))

In [ ]:
# Extract optimized landmarks and poses
def extract_optimized_landmarks(result, max_landmark_id=1000):
    """Extract optimized landmark positions from GTSAM result"""
    landmarks = {}
    for i in range(max_landmark_id):
        Lj = gtsam.symbol('l', i)
        if result.exists(Lj):
            landmark_pos = result.atPoint3(Lj)
            landmarks[i] = np.array(landmark_pos)
    return landmarks

def extract_optimized_poses(result, num_poses=1000):
    """Extract optimized pose positions from GTSAM result"""
    poses = {}
    for i in range(num_poses):
        try:
            Xi = gtsam.symbol('x', i)
            if result.exists(Xi):
                pose = result.atPose3(Xi)
                # Extract translation component
                translation = np.array([pose.x(), pose.y(), pose.z()])
                poses[i] = translation
        except:
            break
    return poses

# Extract optimized results
optimized_landmarks = extract_optimized_landmarks(result)
optimized_poses = extract_optimized_poses(result)

print(f"Found {len(optimized_landmarks)} optimized landmarks")
print(f"Found {len(optimized_poses)} optimized poses")

# Extract unoptimized landmarks (from initial estimates)
unoptimized_landmarks = extract_optimized_landmarks(initial_estimate)
unoptimized_poses = extract_optimized_poses(initial_estimate)

print(f"Found {len(unoptimized_landmarks)} unoptimized landmarks")
print(f"Found {len(unoptimized_poses)} unoptimized poses")


In [ ]:
# 3D Visualization of optimized vs unoptimized results
def visualize_landmarks_and_poses(landmarks_dict, poses_dict, title, color='blue', marker='o', size=50):
    """Visualize landmarks and poses in 3D with interactive rotation"""
    fig = plt.figure(figsize=(15, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot landmarks
    if landmarks_dict:
        landmark_points = np.array(list(landmarks_dict.values()))
        ax.scatter(landmark_points[:, 0], landmark_points[:, 1], landmark_points[:, 2], 
                  c=color, marker=marker, s=size, alpha=0.8, label=f'Landmarks ({len(landmark_points)})')
    
    # Plot poses
    if poses_dict:
        pose_points = np.array(list(poses_dict.values()))
        ax.scatter(pose_points[:, 0], pose_points[:, 1], pose_points[:, 2], 
                  c='red', marker='^', s=30, alpha=0.6, label=f'Poses ({len(pose_points)})')
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'{title} - Interactive 3D Plot (Click and drag to rotate)')
    ax.legend()
    
    # Set equal aspect ratio for better visualization
    all_points = []
    if landmarks_dict:
        all_points.append(np.array(list(landmarks_dict.values())))
    if poses_dict:
        all_points.append(np.array(list(poses_dict.values())))
    
    if all_points:
        # Combine all points to calculate overall bounds
        combined_points = np.vstack(all_points)
        max_range = np.array([combined_points[:, 0].max() - combined_points[:, 0].min(),
                             combined_points[:, 1].max() - combined_points[:, 1].min(),
                             combined_points[:, 2].max() - combined_points[:, 2].min()]).max() / 2.0
        mid_x = (combined_points[:, 0].max() + combined_points[:, 0].min()) * 0.5
        mid_y = (combined_points[:, 1].max() + combined_points[:, 1].min()) * 0.5
        mid_z = (combined_points[:, 2].max() + combined_points[:, 2].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    # Set equal aspect ratio for all axes
    ax.set_box_aspect([1,1,1])  # Equal aspect ratio for 3D plots
    
    # Enable interactive features
    ax.grid(True, alpha=0.3)
    
    # Add some styling for better visibility
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    
    # Make the panes transparent
    ax.xaxis.pane.set_edgecolor('w')
    ax.yaxis.pane.set_edgecolor('w')
    ax.zaxis.pane.set_edgecolor('w')
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax

# Visualize unoptimized results
print("Unoptimized Results:")
fig1, ax1 = visualize_landmarks_and_poses(unoptimized_landmarks, unoptimized_poses, 
                              "Unoptimized Landmarks and Poses", color='lightblue', marker='o')

# Visualize optimized results  
print("Optimized Results:")
fig2, ax2 = visualize_landmarks_and_poses(optimized_landmarks, optimized_poses, 
                              "Optimized Landmarks and Poses", color='darkblue', marker='o')


In [ ]:
# Side-by-side comparison and statistics
def compare_optimization_results(unopt_landmarks, opt_landmarks, unopt_poses, opt_poses):
    """Compare unoptimized vs optimized results with interactive 3D plots"""
    
    # Create side-by-side plot
    fig = plt.figure(figsize=(20, 8))
    
    # Unoptimized plot
    ax1 = fig.add_subplot(121, projection='3d')
    if unopt_landmarks:
        unopt_landmark_points = np.array(list(unopt_landmarks.values()))
        ax1.scatter(unopt_landmark_points[:, 0], unopt_landmark_points[:, 1], unopt_landmark_points[:, 2], 
                   c='lightblue', marker='o', s=50, alpha=0.8, label=f'Landmarks ({len(unopt_landmark_points)})')
    
    if unopt_poses:
        unopt_pose_points = np.array(list(unopt_poses.values()))
        ax1.scatter(unopt_pose_points[:, 0], unopt_pose_points[:, 1], unopt_pose_points[:, 2], 
                   c='red', marker='^', s=30, alpha=0.6, label=f'Poses ({len(unopt_pose_points)})')
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Unoptimized Results - Interactive 3D')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Optimized plot
    ax2 = fig.add_subplot(122, projection='3d')
    if opt_landmarks:
        opt_landmark_points = np.array(list(opt_landmarks.values()))
        ax2.scatter(opt_landmark_points[:, 0], opt_landmark_points[:, 1], opt_landmark_points[:, 2], 
                   c='darkblue', marker='o', s=50, alpha=0.8, label=f'Landmarks ({len(opt_landmark_points)})')
    
    if opt_poses:
        opt_pose_points = np.array(list(opt_poses.values()))
        ax2.scatter(opt_pose_points[:, 0], opt_pose_points[:, 1], opt_pose_points[:, 2], 
                   c='red', marker='^', s=30, alpha=0.6, label=f'Poses ({len(opt_pose_points)})')
    
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Optimized Results - Interactive 3D')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Set equal aspect ratio for both subplots
    for ax, landmarks_dict, poses_dict in [(ax1, unopt_landmarks, unopt_poses), (ax2, opt_landmarks, opt_poses)]:
        # Calculate bounds for equal aspect ratio
        all_points = []
        if landmarks_dict:
            all_points.append(np.array(list(landmarks_dict.values())))
        if poses_dict:
            all_points.append(np.array(list(poses_dict.values())))
        
        if all_points:
            combined_points = np.vstack(all_points)
            max_range = np.array([combined_points[:, 0].max() - combined_points[:, 0].min(),
                                 combined_points[:, 1].max() - combined_points[:, 1].min(),
                                 combined_points[:, 2].max() - combined_points[:, 2].min()]).max() / 2.0
            mid_x = (combined_points[:, 0].max() + combined_points[:, 0].min()) * 0.5
            mid_y = (combined_points[:, 1].max() + combined_points[:, 1].min()) * 0.5
            mid_z = (combined_points[:, 2].max() + combined_points[:, 2].min()) * 0.5
            ax.set_xlim(mid_x - max_range, mid_x + max_range)
            ax.set_ylim(mid_y - max_range, mid_y + max_range)
            ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    # Style both subplots for better visibility
    for ax in [ax1, ax2]:
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('w')
        ax.yaxis.pane.set_edgecolor('w')
        ax.zaxis.pane.set_edgecolor('w')
        # Set equal aspect ratio for all axes
        ax.set_box_aspect([1,1,1])  # Equal aspect ratio for 3D plots
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax1, ax2
    
    # Calculate statistics
    print("\n" + "="*50)
    print("OPTIMIZATION STATISTICS")
    print("="*50)
    
    if unopt_landmarks and opt_landmarks:
        # Find common landmarks
        common_landmark_ids = set(unopt_landmarks.keys()) & set(opt_landmarks.keys())
        if common_landmark_ids:
            print(f"\nLandmark Analysis:")
            print(f"  Total landmarks (unoptimized): {len(unopt_landmarks)}")
            print(f"  Total landmarks (optimized): {len(opt_landmarks)}")
            print(f"  Common landmarks: {len(common_landmark_ids)}")
            
            # Calculate displacement for common landmarks
            displacements = []
            for lid in common_landmark_ids:
                unopt_pos = unopt_landmarks[lid]
                opt_pos = opt_landmarks[lid]
                displacement = np.linalg.norm(opt_pos - unopt_pos)
                displacements.append(displacement)
            
            if displacements:
                print(f"  Mean landmark displacement: {np.mean(displacements):.6f}")
                print(f"  Max landmark displacement: {np.max(displacements):.6f}")
                print(f"  Min landmark displacement: {np.min(displacements):.6f}")
    
    if unopt_poses and opt_poses:
        # Find common poses
        common_pose_ids = set(unopt_poses.keys()) & set(opt_poses.keys())
        if common_pose_ids:
            print(f"\nPose Analysis:")
            print(f"  Total poses (unoptimized): {len(unopt_poses)}")
            print(f"  Total poses (optimized): {len(opt_poses)}")
            print(f"  Common poses: {len(common_pose_ids)}")
            
            # Calculate displacement for common poses
            pose_displacements = []
            for pid in common_pose_ids:
                unopt_pos = unopt_poses[pid]
                opt_pos = opt_poses[pid]
                displacement = np.linalg.norm(opt_pos - unopt_pos)
                pose_displacements.append(displacement)
            
            if pose_displacements:
                print(f"  Mean pose displacement: {np.mean(pose_displacements):.6f}")
                print(f"  Max pose displacement: {np.max(pose_displacements):.6f}")
                print(f"  Min pose displacement: {np.min(pose_displacements):.6f}")

# Run comparison
fig_comp, ax1_comp, ax2_comp = compare_optimization_results(unoptimized_landmarks, optimized_landmarks, 
                           unoptimized_poses, optimized_poses)


In [ ]:
# # Alternative interactive visualization using plotly (if available)
# try:
#     import plotly.graph_objects as go
#     import plotly.express as px
#     from plotly.subplots import make_subplots
    
#     def create_interactive_3d_plot(landmarks_dict, poses_dict, title, color='blue'):
#         """Create highly interactive 3D plot using plotly"""
        
#         fig = go.Figure()
        
#         # Add landmarks
#         if landmarks_dict:
#             landmark_points = np.array(list(landmarks_dict.values()))
#             fig.add_trace(go.Scatter3d(
#                 x=landmark_points[:, 0],
#                 y=landmark_points[:, 1],
#                 z=landmark_points[:, 2],
#                 mode='markers',
#                 marker=dict(
#                     size=8,
#                     color=color,
#                     opacity=0.8
#                 ),
#                 name=f'Landmarks ({len(landmark_points)})',
#                 text=[f'Landmark {i}' for i in landmarks_dict.keys()],
#                 hovertemplate='<b>%{text}</b><br>' +
#                             'X: %{x:.3f}<br>' +
#                             'Y: %{y:.3f}<br>' +
#                             'Z: %{z:.3f}<extra></extra>'
#             ))
        
#         # Add poses
#         if poses_dict:
#             pose_points = np.array(list(poses_dict.values()))
#             fig.add_trace(go.Scatter3d(
#                 x=pose_points[:, 0],
#                 y=pose_points[:, 1],
#                 z=pose_points[:, 2],
#                 mode='markers',
#                 marker=dict(
#                     size=6,
#                     color='red',
#                     symbol='triangle-up',
#                     opacity=0.6
#                 ),
#                 name=f'Poses ({len(pose_points)})',
#                 text=[f'Pose {i}' for i in poses_dict.keys()],
#                 hovertemplate='<b>%{text}</b><br>' +
#                             'X: %{x:.3f}<br>' +
#                             'Y: %{y:.3f}<br>' +
#                             'Z: %{z:.3f}<extra></extra>'
#             ))
        
#         fig.update_layout(
#             title=title,
#             scene=dict(
#                 xaxis_title='X',
#                 yaxis_title='Y',
#                 zaxis_title='Z',
#                 aspectmode='data'  # Equal aspect ratio
#             ),
#             width=800,
#             height=600
#         )
        
#         return fig
    
#     def create_side_by_side_plotly(unopt_landmarks, opt_landmarks, unopt_poses, opt_poses):
#         """Create side-by-side comparison using plotly"""
        
#         # Create subplots
#         fig = make_subplots(
#             rows=1, cols=2,
#             specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
#             subplot_titles=('Unoptimized Results', 'Optimized Results'),
#             horizontal_spacing=0.1
#         )
        
#         # Unoptimized plot
#         if unopt_landmarks:
#             unopt_landmark_points = np.array(list(unopt_landmarks.values()))
#             fig.add_trace(go.Scatter3d(
#                 x=unopt_landmark_points[:, 0],
#                 y=unopt_landmark_points[:, 1],
#                 z=unopt_landmark_points[:, 2],
#                 mode='markers',
#                 marker=dict(size=8, color='lightblue', opacity=0.8),
#                 name='Unopt Landmarks',
#                 showlegend=False
#             ), row=1, col=1)
        
#         if unopt_poses:
#             unopt_pose_points = np.array(list(unopt_poses.values()))
#             fig.add_trace(go.Scatter3d(
#                 x=unopt_pose_points[:, 0],
#                 y=unopt_pose_points[:, 1],
#                 z=unopt_pose_points[:, 2],
#                 mode='markers',
#                 marker=dict(size=6, color='red', symbol='triangle-up', opacity=0.6),
#                 name='Unopt Poses',
#                 showlegend=False
#             ), row=1, col=1)
        
#         # Optimized plot
#         if opt_landmarks:
#             opt_landmark_points = np.array(list(opt_landmarks.values()))
#             fig.add_trace(go.Scatter3d(
#                 x=opt_landmark_points[:, 0],
#                 y=opt_landmark_points[:, 1],
#                 z=opt_landmark_points[:, 2],
#                 mode='markers',
#                 marker=dict(size=8, color='darkblue', opacity=0.8),
#                 name='Opt Landmarks',
#                 showlegend=False
#             ), row=1, col=2)
        
#         if opt_poses:
#             opt_pose_points = np.array(list(opt_poses.values()))
#             fig.add_trace(go.Scatter3d(
#                 x=opt_pose_points[:, 0],
#                 y=opt_pose_points[:, 1],
#                 z=opt_pose_points[:, 2],
#                 mode='markers',
#                 marker=dict(size=6, color='red', symbol='triangle-up', opacity=0.6),
#                 name='Opt Poses',
#                 showlegend=False
#             ), row=1, col=2)
        
#         fig.update_layout(
#             title="GTSAM Optimization Results - Interactive 3D Comparison",
#             width=1200,
#             height=600
#         )
        
#         # Update both subplots to have equal aspect ratio
#         for i in [1, 2]:
#             fig.update_scenes(
#                 aspectmode='data',
#                 row=1, col=i
#             )
        
#         return fig
    
#     print("Creating highly interactive 3D visualizations with plotly...")
    
#     # Individual plots
#     fig_unopt_plotly = create_interactive_3d_plot(unoptimized_landmarks, unoptimized_poses, 
#                                                  "Unoptimized Results - Plotly 3D", 'lightblue')
#     fig_opt_plotly = create_interactive_3d_plot(optimized_landmarks, optimized_poses, 
#                                                "Optimized Results - Plotly 3D", 'darkblue')
    
#     # Side-by-side comparison
#     fig_comparison_plotly = create_side_by_side_plotly(unoptimized_landmarks, optimized_landmarks, 
#                                                       unoptimized_poses, optimized_poses)
    
#     # Show plots
#     print("Unoptimized Results (Plotly):")
#     fig_unopt_plotly.show()
    
#     print("Optimized Results (Plotly):")
#     fig_opt_plotly.show()
    
#     print("Side-by-side Comparison (Plotly):")
#     fig_comparison_plotly.show()
    
# except ImportError:
#     print("Plotly not available. Using matplotlib interactive plots only.")
#     print("To install plotly for better 3D interactivity, run: pip install plotly")
